In [1]:
import os
import re
import copy
import random
from datetime import datetime
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models.resnet import ResNet, BasicBlock
from PIL import Image
from torchinfo import summary
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    recall_score,
)
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

os.environ["WANDB_START_METHOD"] = "thread"
import json
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = secret_value_1
secret_value_0 = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_USERNAME"] = secret_value_0

import wandb

run_start = datetime.now().strftime("%Y-%m-%d_%H-%M")
wandb.login(key=os.getenv("WANDB_API_KEY"))

CLASS_NAMES = [
    #"AsymptomaticControls",
    'Ovarian cancer',
    'Glioma',
    'Pancreatic cancer',
    'Head and neck cancer',
    'Non-small-cell lung cancer'
]


class Config:
    if os.path.exists("/kaggle"):
        DATA_DIR = "/kaggle/input/implatelet-multiclass-resnet/Images"
        SAVE_DIR = "/kaggle/working/weights"
    else:
        DATA_DIR = "KEGG_Pathway_Image/Images"
        SAVE_DIR = "weights_kaggle"
    os.makedirs(SAVE_DIR, exist_ok=True)

    NUM_CLASSES = len(CLASS_NAMES)
    ORIGINAL_SIZE = (373, 259)  # (width, height)

    # --- GRUPY (ID pacjenta) ---
    GROUP_REGEX = r"^(.+?)-TR\d+$"

    # --- PODZIAŁ ---
    OUTER_FOLDS = 5         # 1 fold = ~20% test
    INNER_FOLDS = 4         # z reszty 1/4 = val (~20% całości)
    FOLDS_TO_RUN = [0]       # np. [0, 1, 2, 3, 4] dla pełnej walidacji krzyżowej
    SEED = 42

    # --- MODEL ---
    REMOVE_LAYER4 = True     # mniejszy model (256 cech zamiast 512)
    CONV1_KERNEL = 3
    CONV1_STRIDE = 1         # ablacja: 2
    FC_DROPOUT = 0.3

    # --- TRENING ---
    BATCH_SIZE = 16
    EPOCHS = 60
    PATIENCE = 40             # early stopping po val macro-F1
    LR = 3e-4
    WEIGHT_DECAY = 0.01
    LABEL_SMOOTHING = 0.1
    USE_CLASS_WEIGHTS = True
    PRETRAINED = False

    EXCLUDE_PATTERNS = [r"NKI"]

    # --- AUGMENTACJA (tylko train, bez flipów/rotacji) ---
    AUG_DROP = 0.0          # odsetek pikseli zerowanych
    AUG_SCALE = 0.05        # losowa skala jasności na próbkę

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_kaggle_username():
    return (os.environ.get("KAGGLE_USERNAME")
    or os.environ.get("KAGGLE_USER")
    or "unknown_user")

run = wandb.init(
    entity="pg-ai-team",
    project="ResNet",
    name=get_kaggle_username()+"_ResNet" + run_start,
    config={
        "architecture": "ResNet",
        "dataset": "krzkacz344/implatelet",
        "num_classes": Config.NUM_CLASSES,
        "original_size": Config.ORIGINAL_SIZE,
        "batch_size": Config.BATCH_SIZE,
        "epochs": Config.EPOCHS,
        "lr": Config.LR,
        "weight_decay": Config.WEIGHT_DECAY,
        "label_smoothing": Config.LABEL_SMOOTHING,
        "seed": Config.SEED,
        "device": str(Config.DEVICE),
        "data_dir": Config.DATA_DIR,
        "save_dir": Config.SAVE_DIR,
        "pretrained": Config.PRETRAINED
    },
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ======================================================================
# DATASET
# ======================================================================
class MalignantDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        all_files = sorted(
            f
            for f in os.listdir(root_dir)
            if os.path.isfile(os.path.join(root_dir, f))
            and f.lower().endswith(".png")
            and any(c.lower() in f.lower() for c in CLASS_NAMES)
        )
        if not all_files:
            raise ValueError(f"Brak pasujących obrazów w {root_dir}")

        # --- usunięcie próbek z wykluczonych instytucji (np. NKI) ---
        dropped = [f for f in all_files if self._is_excluded(f)]
        files = [f for f in all_files if not self._is_excluded(f)]
        print(f"Wykluczono {len(dropped)} z {len(all_files)} obrazów "
              f"wg wzorców {Config.EXCLUDE_PATTERNS}")
        if dropped:
            print("  wykluczone wg klas:",
                  dict(Counter(CLASS_NAMES[self._label(f)] for f in dropped)))
        else:
            print("  UWAGA: żaden plik nie pasuje do wzorca - sprawdź EXCLUDE_PATTERNS")
        print("  pozostałe wg prefiksu instytucji:",
              dict(Counter(self._institute(f, self._label(f)) for f in files)))
        if not files:
            raise ValueError("Po filtrowaniu nie zostały żadne obrazy")

        self.file_names = files
        self.labels = np.array([self._label(f) for f in files])
        self.groups = np.array(
            [self._group(f, l) for f, l in zip(files, self.labels)]
        )

        imgs = []
        for f in files:
            img = Image.open(os.path.join(root_dir, f)).convert("RGB").getchannel("R")
            imgs.append(np.asarray(img, dtype=np.uint8))
        shapes = {a.shape for a in imgs}
        assert len(shapes) == 1, f"Obrazy mają różne rozmiary: {shapes}"
        self.images = torch.from_numpy(np.stack(imgs)).unsqueeze(1)

    @staticmethod
    def _label(name):
        for i, c in enumerate(CLASS_NAMES):
            if c.lower() in name.lower():
                return i
        raise ValueError(f"Plik {name} nie pasuje do żadnej klasy")

    @staticmethod
    def _group(name, label):
        stem = os.path.splitext(name)[0]
        sample_id = re.sub(
            re.escape(CLASS_NAMES[label]), "", stem, flags=re.IGNORECASE
        ).strip("_- ")
        if Config.GROUP_REGEX:
            m = re.search(Config.GROUP_REGEX, sample_id)
            if m:
                return m.group(1)
        return sample_id

    @staticmethod
    def _is_excluded(name):
        return any(re.search(p, name, flags=re.IGNORECASE) for p in Config.EXCLUDE_PATTERNS)

    @staticmethod
    def _institute(name, label):
        # np. "AMC-HD-001-TR1139" -> "AMC"
        stem = os.path.splitext(name)[0]
        sample_id = re.sub(
            re.escape(CLASS_NAMES[label]), "", stem, flags=re.IGNORECASE
        ).strip("_- ")
        return re.split(r"[-_]", sample_id)[0]
    
    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        return self.images[idx].float() / 255.0, int(self.labels[idx])


def check_no_leak(groups, tr, va, te):
    s_tr, s_va, s_te = (set(groups[i]) for i in (tr, va, te))
    assert not (s_tr & s_va), "Wyciek grup: train/val"
    assert not (s_tr & s_te), "Wyciek grup: train/test"
    assert not (s_va & s_te), "Wyciek grup: val/test"


# ======================================================================
# MODEL
# ======================================================================
def build_model():
    model = ResNet(BasicBlock, [1, 1, 1, 1])
    model.conv1 = nn.Conv2d(
        1, 64,
        kernel_size=Config.CONV1_KERNEL,
        stride=Config.CONV1_STRIDE,
        padding=Config.CONV1_KERNEL // 2,
        bias=False,
    )
    
    feats = 512
    if Config.REMOVE_LAYER4:
        model.layer4 = nn.Identity()
        feats = 256
    
    model.fc = nn.Sequential(
        nn.Dropout(Config.FC_DROPOUT),
        nn.Linear(feats, Config.NUM_CLASSES)
    )
    model = model.to(Config.DEVICE)

    summary(
        model, 
        input_size=(Config.BATCH_SIZE, 1, Config.ORIGINAL_SIZE[1], Config.ORIGINAL_SIZE[0]),
        device=Config.DEVICE
    )
    return model


class NormalizedModel(nn.Module):
    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(float(mean)))
        self.register_buffer("std", torch.tensor(float(std)))

    def forward(self, x):
        return self.model((x - self.mean) / self.std)


# ======================================================================
# TRENING / EWALUACJA
# ======================================================================
def augment(x):
    keep = (torch.rand_like(x) > Config.AUG_DROP).float()
    scale = 1.0 + Config.AUG_SCALE * torch.randn(x.size(0), 1, 1, 1, device=x.device)
    return x * keep * scale


def safe_auc(labels, probs):
    try:
        return roc_auc_score(
            labels, probs, multi_class="ovr", labels=list(range(Config.NUM_CLASSES))
        )
    except ValueError:
        return float("nan")


@torch.no_grad()
def evaluate(model, loader, mean, std):
    model.eval()
    loss_sum, n = 0.0, 0
    labels, probs = [], []
    for x, y in loader:
        x = (x.to(Config.DEVICE) - mean) / std
        y = y.to(Config.DEVICE)
        out = model(x)
        loss_sum += F.cross_entropy(out, y, reduction="sum").item()
        n += y.size(0)
        probs.append(F.softmax(out, 1).cpu().numpy())
        labels.append(y.cpu().numpy())
    labels = np.concatenate(labels)
    probs = np.concatenate(probs)
    preds = probs.argmax(1)

    # Obliczanie swoistości (Specificity) metodą One-vs-Rest dla klasyfikacji wieloklasowej
    cm = confusion_matrix(labels, preds, labels=list(range(Config.NUM_CLASSES)))
    total_samples = cm.sum()
    specificities = []
    for i in range(Config.NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total_samples - tp - fn - fp
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
    macro_specificity = float(np.mean(specificities))

    return {
        "loss": loss_sum / n,
        "acc": float((preds == labels).mean()),
        "sensitivity": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "specificity": macro_specificity,
        "bacc": balanced_accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "auc": safe_auc(labels, probs),
        "labels": labels,
        "preds": preds,
        "probs": probs,
    }


def export_torchscript(model, mean, std, path):
    wrapped = NormalizedModel(model, mean, std).to(Config.DEVICE).eval()
    w, h = Config.ORIGINAL_SIZE
    example = torch.rand(1, 1, h, w, device=Config.DEVICE)
    with torch.no_grad():
        traced = torch.jit.trace(wrapped, example)
    traced.save(path)
    loaded = torch.jit.load(path, map_location=Config.DEVICE).eval()
    with torch.no_grad():
        diff = (wrapped(example) - loaded(example)).abs().max().item()
    print(f"TorchScript zapisany: {path} | max diff: {diff:.2e}")


def run_fold(ds, tr_idx, va_idx, te_idx, fold):
    set_seed(Config.SEED + fold)
    dev = Config.DEVICE

    xtr = ds.images[tr_idx].float() / 255.0
    mean, std = xtr.mean().item(), xtr.std().item()
    del xtr
    print(f"[fold {fold}] mean={mean:.5f} std={std:.5f}")

    train_loader = DataLoader(
        Subset(ds, tr_idx), batch_size=Config.BATCH_SIZE, shuffle=True, drop_last=True
    )
    val_loader = DataLoader(Subset(ds, va_idx), batch_size=Config.BATCH_SIZE)
    test_loader = DataLoader(Subset(ds, te_idx), batch_size=Config.BATCH_SIZE)

    model = build_model()
    print("parametrów: ", sum(p.numel() for p in model.parameters()))
    
    weight = None
    if Config.USE_CLASS_WEIGHTS:
        counts = np.bincount(ds.labels[tr_idx], minlength=Config.NUM_CLASSES)
        w = counts.sum() / (Config.NUM_CLASSES * np.maximum(counts, 1))
        weight = torch.tensor(w, dtype=torch.float32, device=dev)
    criterion = nn.CrossEntropyLoss(
        weight=weight, label_smoothing=Config.LABEL_SMOOTHING
    )
    optimizer = optim.AdamW(
        model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY
    )
    scheduler = SequentialLR(
    optimizer,
    [LinearLR(optimizer, start_factor=0.1, total_iters=3),
     CosineAnnealingLR(optimizer, T_max=Config.EPOCHS - 3, eta_min=1e-6)],
    milestones=[3],
    )
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.99), use_buffers=True)

    best_acc, best_state, bad = -1.0, None, 0
    for epoch in range(Config.EPOCHS):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(dev), y.to(dev)
            x = (augment(x) - mean) / std
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            ema.update_parameters(model)
            with torch.no_grad():
                tl += F.cross_entropy(out, y, reduction="sum").item()
                tc += (out.argmax(1) == y).sum().item()
                tn += y.size(0)
        scheduler.step()

        val = evaluate(ema.module, val_loader, mean, std)

        run.log({
            "epoch": epoch + 1,
            "train_loss": tl / tn,
            "train_acc": tc / tn,
            "val_loss": val['loss'],
            "val_acc": val['acc'],
            "val_sensitivity": val['sensitivity'],
            "val_specificity": val['specificity'],
            "val_bacc": val['bacc'],
            "val_f1": val['f1'],
            "val_auc": val['auc'],
        })
        
        print(
            f"[fold {fold}] Epoch {epoch + 1:02d}/{Config.EPOCHS} | "
            f"TRAIN loss {tl / tn:.4f} acc {tc / tn:.4f} | "
            f"VAL loss {val['loss']:.4f} acc {val['acc']:.4f} "
            f"Sens {val['sensitivity']:.4f} Spec {val['specificity']:.4f} "
            f"bAcc {val['bacc']:.4f} F1 {val['f1']:.4f} AUC {val['auc']:.4f}"
        )
        if val["bacc"] > best_acc:
            best_acc, bad = val["bacc"], 0
            best_state = copy.deepcopy(ema.module.state_dict())
            torch.save(best_state, os.path.join(Config.SAVE_DIR, f"best_fold{fold}.pt"))
            print(f"    nowy najlepszy model (val bacc {best_acc:.4f})")
        else:
            bad += 1
            if bad >= Config.PATIENCE:
                print(f"    early stopping (brak poprawy przez {Config.PATIENCE} epok)")
                break

    model.load_state_dict(best_state)
    test = evaluate(model, test_loader, mean, std)
    print(f"\n[fold {fold}] TEST loss {test['loss']:.4f} acc {test['acc']:.4f} "
          f"sensitivity {test['sensitivity']:.4f} specificity {test['specificity']:.4f} "
          f"balanced-acc {test['bacc']:.4f} macro-F1 {test['f1']:.4f} AUC {test['auc']:.4f}")
    
    run.log({
        "test_acc": test["acc"],
        "test_sensitivity": test["sensitivity"],
        "test_specificity": test["specificity"],
        "test_bacc": test["bacc"],
        "test_f1": test["f1"],
        "test_auc": test["auc"],
        "test_roc": wandb.plot.roc_curve(test["labels"], test["probs"], labels=CLASS_NAMES)
    })
    
    print(classification_report(
        test["labels"], test["preds"],
        target_names=CLASS_NAMES, labels=list(range(Config.NUM_CLASSES)),
        zero_division=0,
    ))
    print("Macierz pomyłek (wiersze = prawdziwa klasa):")
    print(confusion_matrix(
        test["labels"], test["preds"], labels=list(range(Config.NUM_CLASSES))
    ))

    export_torchscript(
        model, mean, std, os.path.join(Config.SAVE_DIR, f"best_model_traced_fold{fold}.pt")
    )
    return test


# ======================================================================
# MAIN
# ======================================================================
def main():
    print("TRAINING CONFIGURATION")
    print("=" * 60)
    print(f"Data: {Config.DATA_DIR}\nSave: {Config.SAVE_DIR}\nDevice: {Config.DEVICE}")
    print(f"Batch {Config.BATCH_SIZE} | epochs {Config.EPOCHS} | LR {Config.LR} | WD {Config.WEIGHT_DECAY}")
    print("=" * 60)

    set_seed(Config.SEED)
    ds = MalignantDataset(Config.DATA_DIR)
    print(f"Wczytano {len(ds)} obrazów, rozmiar {tuple(ds.images.shape[2:])} (H,W)")

    y, groups = ds.labels, ds.groups
    outer = StratifiedGroupKFold(
        n_splits=Config.OUTER_FOLDS, shuffle=True, random_state=Config.SEED
    )
    outer_splits = list(outer.split(np.zeros(len(y)), y, groups))

    oof_pred = np.full(len(y), -1)
    fold_f1, fold_bacc = [], []
    for fold in Config.FOLDS_TO_RUN:
        tv_idx, te_idx = outer_splits[fold]
        inner = StratifiedGroupKFold(
            n_splits=Config.INNER_FOLDS, shuffle=True, random_state=Config.SEED
        )
        tr_rel, va_rel = next(
            inner.split(np.zeros(len(tv_idx)), y[tv_idx], groups[tv_idx])
        )
        tr_idx, va_idx = tv_idx[tr_rel], tv_idx[va_rel]
        check_no_leak(groups, tr_idx, va_idx, te_idx)
        print(f"\n{'#' * 60}\nFOLD {fold} | train {len(tr_idx)} | val {len(va_idx)} | "
              f"test {len(te_idx)} | grup: "
              f"{len(set(groups[tr_idx]))}/{len(set(groups[va_idx]))}/{len(set(groups[te_idx]))}\n{'#' * 60}")

        test = run_fold(ds, tr_idx, va_idx, te_idx, fold)
        oof_pred[te_idx] = test["preds"]
        fold_f1.append(test["f1"])
        fold_bacc.append(test["bacc"])

    if len(fold_f1) > 1:
        print("\n" + "=" * 60)
        print(f"macro-F1 po foldach: {np.round(fold_f1, 4)} | "
              f"średnia {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
        m = oof_pred >= 0
        print(f"Zbiorczo ({m.sum()} próbek):")
        print(classification_report(
            y[m], oof_pred[m], target_names=CLASS_NAMES,
            labels=list(range(Config.NUM_CLASSES)), zero_division=0,
        ))


if __name__ == "__main__":
    main()

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: krzkacz887 (krzkacz887-politechnika-gda-ska) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


TRAINING CONFIGURATION
Data: /kaggle/input/implatelet-multiclass-resnet/Images
Save: /kaggle/working/weights
Device: cuda
Batch 16 | epochs 60 | LR 0.0003 | WD 0.01
Wykluczono 175 z 1025 obrazów wg wzorców ['NKI']
  wykluczone wg klas: {'Non-small-cell lung cancer': 166, 'Ovarian cancer': 9}
  pozostałe wg prefiksu instytucji: {'MGH': 109, 'Maas': 15, 'Radb': 5, 'TR3675': 1, 'TR3677': 1, 'TR3681': 1, 'TR3684': 1, 'TR3687': 1, 'TR3690': 1, 'TR3696': 1, 'TR3699': 1, 'TR3703': 1, 'TR3707': 1, 'TR3712': 1, 'TR3717': 1, 'TR3723': 1, 'TR3726': 1, 'TR3770': 1, 'TR3809': 1, 'TR3815': 1, 'TR3819': 1, 'TR4795': 1, 'UMCU': 2, 'Vumc': 348, 'TR4068': 1, 'TR4069': 1, 'TR4070': 1, 'TR4071': 1, 'TR4072': 1, 'TR4073': 1, 'TR4074': 1, 'TR4075': 1, 'TR4076': 1, 'TR4077': 1, 'TR4078': 1, 'TR4080': 1, 'TR4088': 1, 'TR4089': 1, 'TR4090': 1, 'TR4091': 1, 'TR4092': 1, 'TR4094': 1, 'TR4125': 1, 'TR4127': 1, 'TR4128': 1, 'TR4131': 1, 'TR4134': 1, 'TR4135': 1, 'TR4136': 1, 'TR4137': 1, 'TR4138': 1, 'TR4167': 1, 